<a href="https://colab.research.google.com/github/jumafernandez/ANN-UNSL/blob/main/notebooks/version_4/notebook_05_evaluacion_semantica_ema_dialog2flow_checkpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05 — Evaluación semántica EMA Dialog2Flow con checkpoints

Versión **ID-only + checkpointable**.

- Todos los insumos se descargan por ID desde Google Drive.
- Las recuperaciones top-10 se guardan por `(variante, índice)`.
- Los juicios LLM se guardan inmediatamente, uno por línea, en JSONL.
- La notebook se puede reanudar sin repetir recuperaciones ni juicios ya hechos.

## 1. Instalación e imports

In [ ]:
!pip install -q openai gdown faiss-cpu pandas numpy tqdm scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 50.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import gc
import shutil
from pathlib import Path
from getpass import getpass
from datetime import datetime
from zoneinfo import ZoneInfo

import gdown
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from IPython.display import display, Markdown

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('IN_COLAB:', IN_COLAB)

Mounted at /content/drive
IN_COLAB: True


## 2. Configuración general

In [ ]:
if IN_COLAB:
    DRIVE_BASE_DIR = Path('/content/drive/MyDrive/Doctorado/Cursos/ANN/TF')
    INPUT_DIR = DRIVE_BASE_DIR / 'data' / 'id_cache_ema_dialog2flow'
    CHECKPOINT_DIR = DRIVE_BASE_DIR / 'resultados' / 'semantic_llm' / 'version_4' / 'checkpoints'
    FINAL_DIR = DRIVE_BASE_DIR / 'resultados' / 'semantic_llm' / 'version_4' / 'final_tables'
else:
    INPUT_DIR = Path('./ann_ema_d2f_inputs')
    CHECKPOINT_DIR = Path('./ann_ema_d2f_checkpoints')
    FINAL_DIR = Path('./ann_ema_d2f_final_tables')

RETRIEVAL_DIR = CHECKPOINT_DIR / 'retrieval_checkpoints'
JUDGMENT_DIR = CHECKPOINT_DIR / 'judgment_checkpoints'
SPLIT_DIR = CHECKPOINT_DIR / 'split'

for d in [INPUT_DIR, CHECKPOINT_DIR, RETRIEVAL_DIR, JUDGMENT_DIR, SPLIT_DIR, FINAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EMBEDDING_NAME = 'dialog2flow-joint-bert-base'
EMBEDDING_LABEL = 'Dialog2Flow'

ALPHA_VALUES = [round(x / 10, 1) for x in range(1, 10)]

def alpha_tag(alpha: float) -> str:
    return f'{alpha:.1f}'.replace('.', '_')

VARIANTES_A_EVALUAR = ['estatico', 'dinamico'] + [f'ema_alpha_{alpha_tag(a)}' for a in ALPHA_VALUES]
INDICES_A_EVALUAR = ['IVF', 'HNSW', 'IVFPQ']

K_EVAL = 10
N_QUERIES_LLM = 100
N_QUERIES_SPLIT = 10000
RANDOM_STATE = 42
QUERY_SAMPLE_SEED = RANDOM_STATE + 100
CONTEXT_WINDOW = 2

N_LIST = 4096
N_PROBE = 64
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 256
PQ_M = 32
PQ_NBITS = 8
N_TRAIN_IVF = 100_000

OPENAI_MODEL = 'gpt-4.1-mini'
TEMPERATURE = 0
SLEEP_BETWEEN_CALLS = 0.2

# Para prueba rápida, descomentar:
# N_QUERIES_LLM = 10
# VARIANTES_A_EVALUAR = ['estatico', 'dinamico', 'ema_alpha_0_5']
# INDICES_A_EVALUAR = ['HNSW']

print('INPUT_DIR:', INPUT_DIR)
print('RETRIEVAL_DIR:', RETRIEVAL_DIR)
print('JUDGMENT_DIR:', JUDGMENT_DIR)
print('SPLIT_DIR:', SPLIT_DIR)
print('FINAL_DIR:', FINAL_DIR)
print('Variantes:', VARIANTES_A_EVALUAR)
print('Índices:', INDICES_A_EVALUAR)
print('N_QUERIES_LLM:', N_QUERIES_LLM)

INPUT_DIR: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow
RETRIEVAL_DIR: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints
JUDGMENT_DIR: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/judgment_checkpoints
SPLIT_DIR: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/split
FINAL_DIR: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/final_tables
Variantes: ['estatico', 'dinamico', 'ema_alpha_0_1', 'ema_alpha_0_2', 'ema_alpha_0_3', 'ema_alpha_0_4', 'ema_alpha_0_5', 'ema_alpha_0_6', 'ema_alpha_0_7', 'ema_alpha_0_8', 'ema_alpha_0_9']
Índices: ['IVF', 'HNSW', 'IVFPQ']
N_QUERIES_LLM: 100


## 3. Manifest ID-only

In [ ]:
FILES = {
    'dialogs-2.0.pkl': '1kRbmvg3NB96pWMUl866GZRrT6Zq9vxcb',
    'ids.npy': '1hWC7nLvSboFHyykjr7VFAEmcvz8re1cY',

    'index_idx_N10000_seed42.npy': '1UECGYQxr7YClV2VpF476u7NrL-xSoR_V',
    'query_idx_N10000_seed42.npy': '1B_pFn9iKXi16CaDcfv1juJ7fUwRqRFb8',

    'embeddings_dialog2flow.npy': '1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW',
    'accumulative_embeddings_dialog2flow.npy': '1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU',

    'ema_embeddings_dialog2flow_alpha_0_1.npy': '1na2SS7OxjMdHwzkKs_HPftbbKUV9NTdX',
    'ema_embeddings_dialog2flow_alpha_0_2.npy': '1j9F_P7CVd0EJ3TTI3cRtTb1bbIXoL4x1',
    'ema_embeddings_dialog2flow_alpha_0_3.npy': '1yqZJUVAo919r45wZWkkADiKdFdr44Sx8',
    'ema_embeddings_dialog2flow_alpha_0_4.npy': '1s9SVF6J1uBhgVWNSJnMqUzAqMjgETyw0',
    'ema_embeddings_dialog2flow_alpha_0_5.npy': '1fr5IJEi8vJaF17D74Rhog1MbieG-3Nt5',
    'ema_embeddings_dialog2flow_alpha_0_6.npy': '1oGqM0cpIkfGacPr3OY-DW-GYrlYMFH4h',
    'ema_embeddings_dialog2flow_alpha_0_7.npy': '1KmtA7MiAUgm2I7NRTy1Rxi9OZPUIGpqh',
    'ema_embeddings_dialog2flow_alpha_0_8.npy': '1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN',
    'ema_embeddings_dialog2flow_alpha_0_9.npy': '1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ',
}

def local_path(filename: str) -> Path:
    return INPUT_DIR / filename

def download_file(filename: str, force: bool = False) -> Path:
    if filename not in FILES:
        raise KeyError(f'Archivo no registrado en FILES: {filename}')
    path = local_path(filename)
    if path.exists() and not force:
        return path
    if path.exists() and force:
        path.unlink()
    file_id = FILES[filename]
    url = f'https://drive.google.com/uc?id={file_id}'
    print('=' * 80)
    print('Descargando por ID')
    print('Archivo:', filename)
    print('ID:', file_id)
    print('Destino:', path)
    gdown.download(url, str(path), quiet=False, fuzzy=True)
    if not path.exists():
        raise FileNotFoundError(f'No se pudo descargar {filename} en {path}')
    return path

def variant_to_filename(variante: str) -> str:
    if variante == 'estatico':
        return 'embeddings_dialog2flow.npy'
    if variante == 'dinamico':
        return 'accumulative_embeddings_dialog2flow.npy'
    if variante.startswith('ema_alpha_'):
        tag = variante.replace('ema_alpha_', '')
        return f'ema_embeddings_dialog2flow_alpha_{tag}.npy'
    raise ValueError(f'Variante no reconocida: {variante}')

def get_embedding_path(variante: str) -> Path:
    return download_file(variant_to_filename(variante), force=False)

def safe_name(text: str) -> str:
    return str(text).replace('/', '_').replace(' ', '_')

## 4. Carga del dataset textual

In [ ]:
ruta_df = download_file('dialogs-2.0.pkl')
ruta_ids = download_file('ids.npy')

df = pd.read_pickle(ruta_df).reset_index(drop=True)
ids = np.load(ruta_ids)

print('DataFrame:', df.shape)
print('IDs:', ids.shape)
display(df.head())

if len(df) != len(ids):
    print('ADVERTENCIA: len(df) != len(ids)')

DataFrame: (1000023, 11)
IDs: (1000023,)


,dataset,split,dialogue_id,turn_id,speaker,utterance,domains,dialog_acts,main_acts,slots,intents
0,ABCD,test,ABCD_test_0,0,user,Hi. My name is Chloe Zhang. I am curious as ...,[storewide query],None,None,None,[timing]
1,ABCD,test,ABCD_test_1,0,user,Hello. I recently signed up for a subscription...,[subscription inquiry],None,None,None,[manage dispute bill]
2,ABCD,test,ABCD_test_1,1,user,"sure, it's Albert Sanders and my account id is...",[subscription inquiry],None,None,[account id],None
3,ABCD,test,ABCD_test_1,2,user,yes its 7149958247,[subscription inquiry],None,None,[order id],None
4,ABCD,test,ABCD_test_2,0,user,I'm thinking about buying an item but first i ...,[single item query],None,None,None,[shirt]


## 5. Verificación opcional de embeddings

In [ ]:
RUN_FULL_DOWNLOAD_CHECK = False

if RUN_FULL_DOWNLOAD_CHECK:
    archivos_necesarios = [
        'embeddings_dialog2flow.npy',
        'accumulative_embeddings_dialog2flow.npy',
    ] + [variant_to_filename(v) for v in VARIANTES_A_EVALUAR if v.startswith('ema_alpha_')]

    for filename in archivos_necesarios:
        path = download_file(filename, force=False)
        arr = np.load(path, mmap_mode='r')
        size_gb = path.stat().st_size / (1024**3)
        print(f'OK {filename:45s} | {size_gb:6.2f} GB | shape={arr.shape} | dtype={arr.dtype}')
        if arr.shape[0] != len(df):
            print('  ADVERTENCIA: cantidad de filas distinta del dataframe')
else:
    print('Verificación completa omitida. Los embeddings se descargarán a demanda.')

Verificación completa omitida. Los embeddings se descargarán a demanda.


## 6. Contexto conversacional para el juez

In [ ]:
_df_ordered = df.reset_index().rename(columns={'index': 'row_id'})

_dialogue_groups = {
    dialogue_id: g.sort_values('turn_id')['row_id'].tolist()
    for dialogue_id, g in _df_ordered.groupby('dialogue_id', sort=False)
}

_row_to_pos = {}
for dialogue_id, rows in _dialogue_groups.items():
    for pos, row_id in enumerate(rows):
        _row_to_pos[int(row_id)] = int(pos)

def get_context_rows(row_id: int, window: int = 2):
    dialogue_id = df.loc[row_id, 'dialogue_id']
    rows = _dialogue_groups[dialogue_id]
    pos = _row_to_pos[int(row_id)]
    start = max(0, pos - window)
    return rows[start:pos + 1]

def format_turn(row_id: int) -> str:
    r = df.loc[row_id]
    speaker = r['speaker'] if 'speaker' in df.columns else '?'
    utterance = str(r['utterance']) if 'utterance' in df.columns else str(r.to_dict())
    turn_id = r['turn_id'] if 'turn_id' in df.columns else row_id
    return f'[{turn_id}] {speaker}: {utterance}'

def format_situation(row_id: int, window: int = 2) -> str:
    rows = get_context_rows(row_id, window=window)
    return '\n'.join(format_turn(rid) for rid in rows)

print(format_situation(0, window=CONTEXT_WINDOW))

[0] user: Hi.  My name is Chloe Zhang.  I am curious as to when my promo code expires. Would you be able to tell me?


## 7. Split desde IDs y query_sample persistente

In [ ]:
index_idx_path = download_file('index_idx_N10000_seed42.npy')
query_idx_path = download_file('query_idx_N10000_seed42.npy')

index_idx = np.load(index_idx_path).astype('int64')
query_idx_full = np.load(query_idx_path).astype('int64')

query_sample_path = SPLIT_DIR / f'query_sample_N{N_QUERIES_LLM}_seed{QUERY_SAMPLE_SEED}.npy'

if query_sample_path.exists():
    query_sample = np.load(query_sample_path).astype('int64')
    print('query_sample cargado:', query_sample_path)
else:
    rng = np.random.default_rng(QUERY_SAMPLE_SEED)
    query_sample = rng.choice(
        query_idx_full,
        size=min(N_QUERIES_LLM, len(query_idx_full)),
        replace=False,
    ).astype('int64')
    np.save(query_sample_path, query_sample)
    print('query_sample generado y guardado:', query_sample_path)

print('Vectores indexados:', len(index_idx))
print('Consultas split completo:', len(query_idx_full))
print('Consultas para LLM:', len(query_sample))
print('Primeras consultas:', query_sample[:10])

Descargando por ID
Archivo: index_idx_N10000_seed42.npy
ID: 1UECGYQxr7YClV2VpF476u7NrL-xSoR_V
Destino: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/index_idx_N10000_seed42.npy


Downloading...
From: https://drive.google.com/uc?id=1UECGYQxr7YClV2VpF476u7NrL-xSoR_V
To: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/index_idx_N10000_seed42.npy
100%|██████████| 7.92M/7.92M [00:00<00:00, 43.8MB/s]


Descargando por ID
Archivo: query_idx_N10000_seed42.npy
ID: 1B_pFn9iKXi16CaDcfv1juJ7fUwRqRFb8
Destino: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/query_idx_N10000_seed42.npy


Downloading...
From: https://drive.google.com/uc?id=1B_pFn9iKXi16CaDcfv1juJ7fUwRqRFb8
To: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/query_idx_N10000_seed42.npy
100%|██████████| 80.1k/80.1k [00:00<00:00, 32.9MB/s]

query_sample generado y guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/split/query_sample_N100_seed142.npy
Vectores indexados: 990023
Consultas split completo: 10000
Consultas para LLM: 100
Primeras consultas: [380462  73765 947093 532607 956850 877671  57977 553876 643250  76028]


## 8. Recuperación top-10 con checkpoints por configuración

In [ ]:
def retrieval_checkpoint_path(variante: str, index_type: str) -> Path:
    return RETRIEVAL_DIR / f'retrieval__{safe_name(variante)}__{safe_name(index_type)}.csv'

def normalize_copy(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype='float32').copy()
    faiss.normalize_L2(x)
    return x

def build_ann_index(index_type: str, index_vectors: np.ndarray, random_state: int = 42):
    n, d = index_vectors.shape
    if index_type == 'IVF':
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, N_LIST, faiss.METRIC_L2)
        rng = np.random.default_rng(random_state)
        train_size = min(N_TRAIN_IVF, n)
        train_idx = rng.choice(n, size=train_size, replace=False)
        index.train(index_vectors[train_idx])
        index.add(index_vectors)
        index.nprobe = N_PROBE
        return index
    if index_type == 'HNSW':
        index = faiss.IndexHNSWFlat(d, HNSW_M)
        index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
        index.add(index_vectors)
        index.hnsw.efSearch = HNSW_EF_SEARCH
        return index
    if index_type == 'IVFPQ':
        if d % PQ_M != 0:
            raise ValueError(f'La dimensión {d} no es divisible por PQ_M={PQ_M}')
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFPQ(quantizer, d, N_LIST, PQ_M, PQ_NBITS)
        rng = np.random.default_rng(random_state)
        train_size = min(N_TRAIN_IVF, n)
        train_idx = rng.choice(n, size=train_size, replace=False)
        index.train(index_vectors[train_idx])
        index.add(index_vectors)
        index.nprobe = N_PROBE
        return index
    raise ValueError(f'Índice no reconocido: {index_type}')

def retrieve_topk_for_config(variante: str, index_type: str, k: int = 10) -> pd.DataFrame:
    ckpt_path = retrieval_checkpoint_path(variante, index_type)
    if ckpt_path.exists():
        print(f'Checkpoint encontrado: {ckpt_path}')
        return pd.read_csv(ckpt_path)

    path = get_embedding_path(variante)
    matriz = np.load(path, mmap_mode='r')

    print('=' * 80)
    print('Construyendo recuperación')
    print('Variante:', variante, '| Índice:', index_type)
    print('Archivo:', path.name, '| Shape:', matriz.shape)

    index_vectors = normalize_copy(matriz[index_idx])
    query_vectors = normalize_copy(matriz[query_sample])

    t0 = time.time()
    index = build_ann_index(index_type, index_vectors, random_state=RANDOM_STATE)
    build_time = time.time() - t0

    t1 = time.time()
    distances, local_neighbors = index.search(query_vectors, k)
    search_time = time.time() - t1

    neighbor_rows = index_idx[local_neighbors]

    records = []
    for q_pos, q_row in enumerate(query_sample):
        q_row = int(q_row)
        for rank in range(k):
            n_row = int(neighbor_rows[q_pos, rank])
            records.append({
                'embedding_name': EMBEDDING_NAME,
                'embedding_base': EMBEDDING_LABEL,
                'variant': variante,
                'index_type': index_type,
                'query_order': int(q_pos),
                'query_row': q_row,
                'neighbor_rank': int(rank + 1),
                'neighbor_row': n_row,
                'distance': float(distances[q_pos, rank]),
                'query_utterance': str(df.loc[q_row, 'utterance']),
                'neighbor_utterance': str(df.loc[n_row, 'utterance']),
                'query_context': format_situation(q_row, window=CONTEXT_WINDOW),
                'neighbor_context': format_situation(n_row, window=CONTEXT_WINDOW),
                'build_time_s': build_time,
                'search_time_s': search_time,
            })

    df_part = pd.DataFrame(records)
    tmp_path = ckpt_path.with_suffix('.tmp.csv')
    df_part.to_csv(tmp_path, index=False)
    os.replace(tmp_path, ckpt_path)
    print('Checkpoint guardado:', ckpt_path)

    del matriz, index_vectors, query_vectors, index, distances, local_neighbors, neighbor_rows
    gc.collect()
    return df_part

def run_all_retrievals_checkpointed():
    parts = []
    total = len(VARIANTES_A_EVALUAR) * len(INDICES_A_EVALUAR)
    done = 0
    for variante in VARIANTES_A_EVALUAR:
        for index_type in INDICES_A_EVALUAR:
            done += 1
            print(f'\n[{done}/{total}] {variante} / {index_type}')
            parts.append(retrieve_topk_for_config(variante, index_type, k=K_EVAL))
    return pd.concat(parts, ignore_index=True)

df_retrieval = run_all_retrievals_checkpointed()
print('Recuperaciones totales:', df_retrieval.shape)
display(df_retrieval.head())


[1/33] estatico / IVF
Descargando por ID
Archivo: embeddings_dialog2flow.npy
ID: 1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW
Destino: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/embeddings_dialog2flow.npy


Downloading...
From (original): https://drive.google.com/uc?id=1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW
From (redirected): https://drive.google.com/uc?id=1Pxf6oho0HYwv3B8VObZ_R6asShyK1WFW&confirm=t&uuid=247ebd5b-01aa-4aab-95e4-5ce5b05a4817
To: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/embeddings_dialog2flow.npy
100%|██████████| 3.07G/3.07G [00:48<00:00, 64.0MB/s]


Construyendo recuperación
Variante: estatico | Índice: IVF
Archivo: embeddings_dialog2flow.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__estatico__IVF.csv

[2/33] estatico / HNSW
Construyendo recuperación
Variante: estatico | Índice: HNSW
Archivo: embeddings_dialog2flow.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__estatico__HNSW.csv

[3/33] estatico / IVFPQ
Construyendo recuperación
Variante: estatico | Índice: IVFPQ
Archivo: embeddings_dialog2flow.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__estatico__IVFPQ.csv

[4/33] dinamico / IVF
Descargando por ID
Archivo: accumulative_embeddings_dialog2flow.npy
ID

Downloading...
From (original): https://drive.google.com/uc?id=1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU
From (redirected): https://drive.google.com/uc?id=1ywEjHOKNRWkq7YbzmCzP-JT4DUgm6zAU&confirm=t&uuid=9226e569-ee65-478d-899f-e46c3046e7bb
To: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/accumulative_embeddings_dialog2flow.npy
100%|██████████| 3.07G/3.07G [00:55<00:00, 54.9MB/s]


Construyendo recuperación
Variante: dinamico | Índice: IVF
Archivo: accumulative_embeddings_dialog2flow.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__dinamico__IVF.csv

[5/33] dinamico / HNSW
Construyendo recuperación
Variante: dinamico | Índice: HNSW
Archivo: accumulative_embeddings_dialog2flow.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__dinamico__HNSW.csv

[6/33] dinamico / IVFPQ
Construyendo recuperación
Variante: dinamico | Índice: IVFPQ
Archivo: accumulative_embeddings_dialog2flow.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__dinamico__IVFPQ.csv

[7/33] ema_alpha_0_1 / IVF
Descargando por ID
Archivo

...

Downloading...
From (original): https://drive.google.com/uc?id=1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN
From (redirected): https://drive.google.com/uc?id=1UFR-nSi1J37IRVSe30GvxKWoW5bUULcN&confirm=t&uuid=7fc3dcb3-99da-45fd-a28c-0dfb0817cd76
To: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/ema_embeddings_dialog2flow_alpha_0_8.npy
100%|██████████| 3.07G/3.07G [01:07<00:00, 45.6MB/s]


Construyendo recuperación
Variante: ema_alpha_0_8 | Índice: IVF
Archivo: ema_embeddings_dialog2flow_alpha_0_8.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__ema_alpha_0_8__IVF.csv

[29/33] ema_alpha_0_8 / HNSW
Construyendo recuperación
Variante: ema_alpha_0_8 | Índice: HNSW
Archivo: ema_embeddings_dialog2flow_alpha_0_8.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__ema_alpha_0_8__HNSW.csv

[30/33] ema_alpha_0_8 / IVFPQ
Construyendo recuperación
Variante: ema_alpha_0_8 | Índice: IVFPQ
Archivo: ema_embeddings_dialog2flow_alpha_0_8.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__ema_alpha_0_8__IVFPQ.csv

[31/33] 

Downloading...
From (original): https://drive.google.com/uc?id=1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ
From (redirected): https://drive.google.com/uc?id=1hN-AED1dO64hYDk_nccyqqXgy2VtCJZZ&confirm=t&uuid=2a9ec540-7942-469a-9510-49af5d5b2541
To: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/data/id_cache_ema_dialog2flow/ema_embeddings_dialog2flow_alpha_0_9.npy
100%|██████████| 3.07G/3.07G [01:01<00:00, 49.9MB/s]


Construyendo recuperación
Variante: ema_alpha_0_9 | Índice: IVF
Archivo: ema_embeddings_dialog2flow_alpha_0_9.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__ema_alpha_0_9__IVF.csv

[32/33] ema_alpha_0_9 / HNSW
Construyendo recuperación
Variante: ema_alpha_0_9 | Índice: HNSW
Archivo: ema_embeddings_dialog2flow_alpha_0_9.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__ema_alpha_0_9__HNSW.csv

[33/33] ema_alpha_0_9 / IVFPQ
Construyendo recuperación
Variante: ema_alpha_0_9 | Índice: IVFPQ
Archivo: ema_embeddings_dialog2flow_alpha_0_9.npy | Shape: (1000023, 768)
Checkpoint guardado: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/retrieval_checkpoints/retrieval__ema_alpha_0_9__IVFPQ.csv
Recuperac

,embedding_name,embedding_base,variant,index_type,query_order,query_row,neighbor_rank,neighbor_row,distance,query_utterance,neighbor_utterance,query_context,neighbor_context,build_time_s,search_time_s
0,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,380462,1,940881,0.000000,"Okay. I'm showing 3 flight options for you, in...","Okay. I'm showing 3 flight options for you, in...","[5] system: And, what's the city you want for ...","[5] system: And, what's the city you want for ...",170.435355,0.255849
1,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,380462,2,854458,0.075209,"Okay. I'm showing 3 flight options for you, in...","Alright. I have 3 suitable flights for you, wi...","[5] system: And, what's the city you want for ...","[3] system: Okay, when will you be leaving?\n[...",170.435355,0.255849
2,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,380462,3,863917,0.083111,"Okay. I'm showing 3 flight options for you, in...",I have 4 options for you. The first is with Am...,"[5] system: And, what's the city you want for ...",[3] system: Where is your intended destination...,170.435355,0.255849
3,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,380462,4,852347,0.088898,"Okay. I'm showing 3 flight options for you, in...",There are 3 flights. One is with Delta Airline...,"[5] system: And, what's the city you want for ...",[9] system: It arrives at 0:51 am.\n[10] user:...,170.435355,0.255849
4,dialog2flow-joint-bert-base,Dialog2Flow,estatico,IVF,0,380462,5,419302,0.089346,"Okay. I'm showing 3 flight options for you, in...",You have 5 possible options for your flight. O...,"[5] system: And, what's the city you want for ...",[3] system: And what is your departure date?\n...,170.435355,0.255849


## 9. Inspección rápida

In [ ]:
def mostrar_vecinos(df_retrieval, variante='ema_alpha_0_5', index_type='HNSW', query_order=0, top_n=10):
    sub = df_retrieval[
        (df_retrieval['variant'] == variante) &
        (df_retrieval['index_type'] == index_type) &
        (df_retrieval['query_order'] == query_order)
    ].sort_values('neighbor_rank').head(top_n)
    if sub.empty:
        print('No hay registros para ese filtro.')
        return
    first = sub.iloc[0]
    display(Markdown(f'### Consulta `{query_order}` — {variante} / {index_type}'))
    display(Markdown('**Situación de consulta:**'))
    print(first['query_context'])
    display(sub[['neighbor_rank', 'distance', 'neighbor_row', 'neighbor_utterance']])

mostrar_vecinos(df_retrieval, variante='ema_alpha_0_5', index_type='HNSW', query_order=0)

### Consulta `0` — ema_alpha_0_5 / HNSW

**Situación de consulta:**

[5] system: And, what's the city you want for your departure?
[6] user: I want to fly out of Atlanta, GA.
[7] system: Okay. I'm showing 3 flight options for you, including a flight on Delta Airlines with 0 stops, departing at 6:50 am for $260.


,neighbor_rank,distance,neighbor_row,neighbor_utterance
19000,1,6.906024e-14,940881,"Okay. I'm showing 3 flight options for you, in..."
19001,2,1.018683e-01,940880,"I want to fly out of Atlanta, GA."
19002,3,1.018683e-01,380461,"I want to fly out of Atlanta, GA."
19003,4,1.146481e-01,940882,What's the destination airport and arrival tim...
19004,5,1.146481e-01,380463,What's the destination airport and arrival tim...
19005,6,1.201017e-01,852742,"I have locate 1 flight with Alaska Airlines, i..."
19006,7,1.274708e-01,431242,I've found 5 flights that'll fit your needs. T...
19007,8,1.394682e-01,933744,I have 4 flights that match. There's a 0 stop ...
19008,9,1.423699e-01,853160,I found 4 flights for you. Would you like to f...
19009,10,1.478643e-01,893920,"Ok, I found an American Airlines flight with 1..."


## 10. OpenAI

In [ ]:
from openai import OpenAI

os.environ['OPENAI_API_KEY'] = "sk-..."

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
print('Cliente OpenAI configurado.')

Cliente OpenAI configurado.


## 11. Prompt juez

In [ ]:
EVALUATION_SCHEMA = {
    'name': 'dialog_retrieval_evaluation',
    'schema': {
        'type': 'object',
        'additionalProperties': False,
        'properties': {
            'evaluations': {
                'type': 'array',
                'minItems': K_EVAL,
                'maxItems': K_EVAL,
                'items': {
                    'type': 'object',
                    'additionalProperties': False,
                    'properties': {
                        'rank': {'type': 'integer', 'minimum': 1, 'maximum': K_EVAL},
                        'semantic_similarity': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'functional_similarity': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'memory_usefulness': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'overall_similarity': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                        'brief_reason': {'type': 'string'},
                    },
                    'required': [
                        'rank',
                        'semantic_similarity',
                        'functional_similarity',
                        'memory_usefulness',
                        'overall_similarity',
                        'brief_reason',
                    ],
                },
            }
        },
        'required': ['evaluations'],
    },
    'strict': True,
}

SYSTEM_PROMPT = (
    'You are an expert evaluator of task-oriented dialogue retrieval.\n'
    'Your task is to judge whether retrieved dialogue situations are semantically and functionally similar to a query dialogue situation.\n'
    'Focus on task-oriented dialogue behavior, not only lexical overlap.\n'
    'Use the 1-5 scale consistently.\n'
    'Return only valid JSON following the schema.'
)

def build_judge_prompt(group: pd.DataFrame) -> str:
    group = group.sort_values('neighbor_rank')
    first = group.iloc[0]
    lines = []
    lines.append('Evaluate whether each retrieved neighbor is similar to the query situation.')
    lines.append('')
    lines.append('Scoring scale:')
    lines.append('1 = unrelated')
    lines.append('2 = weak or superficial relation')
    lines.append('3 = partial similarity')
    lines.append('4 = clear semantic/functional similarity')
    lines.append('5 = highly equivalent dialogue situations')
    lines.append('')
    lines.append('QUERY SITUATION:')
    lines.append(first['query_context'])
    lines.append('')
    lines.append('RETRIEVED NEIGHBORS:')
    for _, r in group.iterrows():
        lines.append('')
        lines.append(f"Neighbor rank {int(r['neighbor_rank'])}:")
        lines.append(r['neighbor_context'])
    lines.append('')
    lines.append(f'Return one evaluation object for each neighbor rank from 1 to {K_EVAL}.')
    return '\n'.join(lines)

def evaluate_group_with_llm(group: pd.DataFrame, model: str = OPENAI_MODEL) -> dict:
    user_prompt = build_judge_prompt(group)
    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        response_format={
            'type': 'json_schema',
            'json_schema': EVALUATION_SCHEMA,
        },
    )
    return json.loads(response.choices[0].message.content)

_test_key, _test_group = next(iter(df_retrieval.groupby(['embedding_base', 'variant', 'index_type', 'query_order'])))
print(build_judge_prompt(_test_group)[:2500])

Evaluate whether each retrieved neighbor is similar to the query situation.

Scoring scale:
1 = unrelated
2 = weak or superficial relation
3 = partial similarity
4 = clear semantic/functional similarity
5 = highly equivalent dialogue situations

QUERY SITUATION:
[5] system: And, what's the city you want for your departure?
[6] user: I want to fly out of Atlanta, GA.
[7] system: Okay. I'm showing 3 flight options for you, including a flight on Delta Airlines with 0 stops, departing at 6:50 am for $260.

RETRIEVED NEIGHBORS:

Neighbor rank 1:
[5] system: And, what's the city you want for your departure?
[6] user: I want to fly out of Atlanta, GA.
[7] system: Okay. I'm showing 3 flight options for you, including a flight on Delta Airlines with 0 stops, departing at 6:50 am for $260.

Neighbor rank 2:
[4] user: I'd like to fly out on the 13th of this month.
[5] system: And, what's the city you want for your departure?
[6] user: I want to fly out of Atlanta, GA.

Neighbor rank 3:
[4] user: 

## 12. Juicios LLM con checkpoints por configuración

In [ ]:
def judgment_checkpoint_path(variante: str, index_type: str) -> Path:
    return JUDGMENT_DIR / f'judgments__{safe_name(variante)}__{safe_name(index_type)}.jsonl'

def judgment_key(variante: str, index_type: str, query_order: int, query_row: int) -> str:
    return '|'.join([
        EMBEDDING_LABEL,
        str(variante),
        str(index_type),
        str(query_order),
        str(query_row),
        OPENAI_MODEL,
    ])

def load_done_judgments(path: Path) -> set:
    done = set()
    if not path.exists():
        return done
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                done.add(rec['config_key'])
            except Exception:
                pass
    return done

def append_jsonl_durable(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush()
        os.fsync(f.fileno())

def run_llm_for_config(df_config: pd.DataFrame, variante: str, index_type: str):
    out_path = judgment_checkpoint_path(variante, index_type)
    done = load_done_judgments(out_path)
    groups = list(df_config.groupby('query_order', sort=False))
    print('=' * 80)
    print('LLM config:', variante, '/', index_type)
    print('Archivo:', out_path)
    print('Queries:', len(groups))
    print('Ya evaluadas:', len(done))
    for query_order, group in tqdm(groups):
        group = group.sort_values('neighbor_rank')
        query_row = int(group.iloc[0]['query_row'])
        config_key = judgment_key(variante, index_type, int(query_order), query_row)
        if config_key in done:
            continue
        try:
            result = evaluate_group_with_llm(group, model=OPENAI_MODEL)
            record = {
                'config_key': config_key,
                'embedding_base': EMBEDDING_LABEL,
                'variant': variante,
                'index_type': index_type,
                'query_order': int(query_order),
                'query_row': query_row,
                'model': OPENAI_MODEL,
                'evaluations': result['evaluations'],
            }
            append_jsonl_durable(out_path, record)
            done.add(config_key)
            time.sleep(SLEEP_BETWEEN_CALLS)
        except Exception as e:
            print('Error en', variante, index_type, query_order, '->', repr(e))
            time.sleep(2)
    print('Finalizado:', variante, '/', index_type)

def run_all_llm_checkpointed(df_retrieval: pd.DataFrame):
    total = len(VARIANTES_A_EVALUAR) * len(INDICES_A_EVALUAR)
    done_cfg = 0
    for variante in VARIANTES_A_EVALUAR:
        for index_type in INDICES_A_EVALUAR:
            done_cfg += 1
            print(f'\n[{done_cfg}/{total}] Evaluación LLM: {variante} / {index_type}')
            df_config = df_retrieval[
                (df_retrieval['variant'] == variante) &
                (df_retrieval['index_type'] == index_type)
            ].copy()
            if df_config.empty:
                print('Sin retrieval para esta configuración. Se omite.')
                continue
            run_llm_for_config(df_config, variante, index_type)

run_all_llm_checkpointed(df_retrieval)


[1/33] Evaluación LLM: estatico / IVF
LLM config: estatico / IVF
Archivo: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/judgment_checkpoints/judgments__estatico__IVF.jsonl
Queries: 100
Ya evaluadas: 0


  0%|          | 0/100 [00:00<?, ?it/s]

Finalizado: estatico / IVF

[2/33] Evaluación LLM: estatico / HNSW
LLM config: estatico / HNSW
Archivo: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/judgment_checkpoints/judgments__estatico__HNSW.jsonl
Queries: 100
Ya evaluadas: 0


  0%|          | 0/100 [00:00<?, ?it/s]

Finalizado: estatico / HNSW

[3/33] Evaluación LLM: estatico / IVFPQ
LLM config: estatico / IVFPQ
Archivo: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/judgment_checkpoints/judgments__estatico__IVFPQ.jsonl
Queries: 100
Ya evaluadas: 0


...

Finalizado: ema_alpha_0_9 / IVF

[32/33] Evaluación LLM: ema_alpha_0_9 / HNSW
LLM config: ema_alpha_0_9 / HNSW
Archivo: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/judgment_checkpoints/judgments__ema_alpha_0_9__HNSW.jsonl
Queries: 100
Ya evaluadas: 0


  0%|          | 0/100 [00:00<?, ?it/s]

Finalizado: ema_alpha_0_9 / HNSW

[33/33] Evaluación LLM: ema_alpha_0_9 / IVFPQ
LLM config: ema_alpha_0_9 / IVFPQ
Archivo: /content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/checkpoints/judgment_checkpoints/judgments__ema_alpha_0_9__IVFPQ.jsonl
Queries: 100
Ya evaluadas: 0


  0%|          | 0/100 [00:00<?, ?it/s]

Finalizado: ema_alpha_0_9 / IVFPQ


## 13. Consolidación de juicios

In [ ]:
def read_all_judgments() -> pd.DataFrame:
    records = []
    files = sorted(JUDGMENT_DIR.glob('judgments__*.jsonl'))
    print('Archivos de juicios encontrados:', len(files))
    for path in files:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                for ev in rec['evaluations']:
                    records.append({
                        'embedding_base': rec['embedding_base'],
                        'variant': rec['variant'],
                        'index_type': rec['index_type'],
                        'query_order': rec['query_order'],
                        'query_row': rec['query_row'],
                        'model': rec['model'],
                        **ev,
                    })
    if not records:
        return pd.DataFrame()
    scores = pd.DataFrame(records)
    scores = scores.rename(columns={'rank': 'neighbor_rank'})
    return scores

df_scores_raw = read_all_judgments()
print('Scores raw:', df_scores_raw.shape)
display(df_scores_raw.head())

merge_cols = [
    'embedding_base', 'variant', 'index_type',
    'query_order', 'query_row', 'neighbor_rank',
]

df_scores = df_scores_raw.merge(
    df_retrieval,
    on=merge_cols,
    how='left',
    validate='many_to_one',
)

print('Scores enriquecidos:', df_scores.shape)
display(df_scores.head())

Archivos de juicios encontrados: 33
Scores raw: (33000, 12)


,embedding_base,variant,index_type,query_order,query_row,model,neighbor_rank,semantic_similarity,functional_similarity,memory_usefulness,overall_similarity,brief_reason
0,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,1,5,5,5,5,Exact same dialogue with identical user and sy...
1,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,2,4,4,4,4,"Same departure city question and answer, but m..."
2,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,3,4,4,4,4,"Same as neighbor 2, repeated; clear semantic a..."
3,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,4,4,4,4,4,Includes user departure city and system flight...
4,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,5,4,4,4,4,"Same as neighbor 4, repeated; maintains clear ..."


Scores enriquecidos: (33000, 21)


,embedding_base,variant,index_type,query_order,query_row,model,neighbor_rank,semantic_similarity,functional_similarity,memory_usefulness,...,brief_reason,embedding_name,neighbor_row,distance,query_utterance,neighbor_utterance,query_context,neighbor_context,build_time_s,search_time_s
0,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,1,5,5,5,...,Exact same dialogue with identical user and sy...,dialog2flow-joint-bert-base,940881,6.884739e-14,"Okay. I'm showing 3 flight options for you, in...","Okay. I'm showing 3 flight options for you, in...","[5] system: And, what's the city you want for ...","[5] system: And, what's the city you want for ...",906.860261,0.102842
1,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,2,4,4,4,...,"Same departure city question and answer, but m...",dialog2flow-joint-bert-base,940880,1.018667e-01,"Okay. I'm showing 3 flight options for you, in...","I want to fly out of Atlanta, GA.","[5] system: And, what's the city you want for ...",[4] user: I'd like to fly out on the 13th of t...,906.860261,0.102842
2,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,3,4,4,4,...,"Same as neighbor 2, repeated; clear semantic a...",dialog2flow-joint-bert-base,380461,1.018667e-01,"Okay. I'm showing 3 flight options for you, in...","I want to fly out of Atlanta, GA.","[5] system: And, what's the city you want for ...",[4] user: I'd like to fly out on the 13th of t...,906.860261,0.102842
3,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,4,4,4,4,...,Includes user departure city and system flight...,dialog2flow-joint-bert-base,940882,1.146467e-01,"Okay. I'm showing 3 flight options for you, in...",What's the destination airport and arrival tim...,"[5] system: And, what's the city you want for ...","[6] user: I want to fly out of Atlanta, GA.\n[...",906.860261,0.102842
4,Dialog2Flow,dinamico,HNSW,0,380462,gpt-4.1-mini,5,4,4,4,...,"Same as neighbor 4, repeated; maintains clear ...",dialog2flow-joint-bert-base,380463,1.146467e-01,"Okay. I'm showing 3 flight options for you, in...",What's the destination airport and arrival tim...,"[5] system: And, what's the city you want for ...","[6] user: I want to fly out of Atlanta, GA.\n[...",906.860261,0.102842


## 14. Estado de avance

In [ ]:
expected_groups = []
for variante in VARIANTES_A_EVALUAR:
    for index_type in INDICES_A_EVALUAR:
        expected_groups.append((EMBEDDING_LABEL, variante, index_type))

progress_rows = []
for embedding_base, variante, index_type in expected_groups:
    expected_queries = len(query_sample)
    path = judgment_checkpoint_path(variante, index_type)
    done = load_done_judgments(path)
    progress_rows.append({
        'embedding_base': embedding_base,
        'variant': variante,
        'index_type': index_type,
        'expected_queries': expected_queries,
        'done_queries': len(done),
        'pending_queries': max(0, expected_queries - len(done)),
        'complete': len(done) >= expected_queries,
        'path': str(path),
    })

progress_df = pd.DataFrame(progress_rows)
display(progress_df)
print('Total queries esperadas:', progress_df['expected_queries'].sum())
print('Total queries evaluadas:', progress_df['done_queries'].sum())
print('Total queries pendientes:', progress_df['pending_queries'].sum())

,embedding_base,variant,index_type,expected_queries,done_queries,pending_queries,complete,path
0,Dialog2Flow,estatico,IVF,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
1,Dialog2Flow,estatico,HNSW,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
2,Dialog2Flow,estatico,IVFPQ,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
3,Dialog2Flow,dinamico,IVF,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
4,Dialog2Flow,dinamico,HNSW,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
5,Dialog2Flow,dinamico,IVFPQ,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
6,Dialog2Flow,ema_alpha_0_1,IVF,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
7,Dialog2Flow,ema_alpha_0_1,HNSW,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
8,Dialog2Flow,ema_alpha_0_1,IVFPQ,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...
9,Dialog2Flow,ema_alpha_0_2,IVF,100,100,0,True,/content/drive/MyDrive/Doctorado/Cursos/ANN/TF...


Total queries esperadas: 3300
Total queries evaluadas: 3300
Total queries pendientes: 0


## 15. Métricas MSS@10

In [ ]:
if df_scores.empty:
    raise ValueError('No hay juicios consolidados. Ejecutá la evaluación LLM o verificá JUDGMENT_DIR.')

query_metrics = (
    df_scores
    .groupby(['embedding_base', 'variant', 'index_type', 'query_order'], as_index=False)
    .agg(
        semantic_score_at10=('semantic_similarity', 'mean'),
        functional_score_at10=('functional_similarity', 'mean'),
        memory_score_at10=('memory_usefulness', 'mean'),
        mss_at10=('overall_similarity', 'mean'),
    )
)

summary_metrics = (
    query_metrics
    .groupby(['embedding_base', 'variant', 'index_type'], as_index=False)
    .agg(
        n_queries=('query_order', 'nunique'),
        mean_semantic_score_global_at10=('mss_at10', 'mean'),
        sd_semantic_score_global_at10=('mss_at10', 'std'),
        mean_semantic_score_at10=('semantic_score_at10', 'mean'),
        sd_semantic_score_at10=('semantic_score_at10', 'std'),
        mean_functional_score_at10=('functional_score_at10', 'mean'),
        sd_functional_score_at10=('functional_score_at10', 'std'),
        mean_memory_score_at10=('memory_score_at10', 'mean'),
        sd_memory_score_at10=('memory_score_at10', 'std'),
    )
)

summary_metrics['se_semantic_score_global_at10'] = summary_metrics['sd_semantic_score_global_at10'] / np.sqrt(summary_metrics['n_queries'])
summary_metrics['ci95_semantic_score_global_at10'] = 1.96 * summary_metrics['se_semantic_score_global_at10']

for col in summary_metrics.select_dtypes(include=['float']).columns:
    summary_metrics[col] = summary_metrics[col].round(3)

display(summary_metrics)

,embedding_base,variant,index_type,n_queries,mean_semantic_score_global_at10,sd_semantic_score_global_at10,mean_semantic_score_at10,sd_semantic_score_at10,mean_functional_score_at10,sd_functional_score_at10,mean_memory_score_at10,sd_memory_score_at10,se_semantic_score_global_at10,ci95_semantic_score_global_at10
0,Dialog2Flow,dinamico,HNSW,100,3.661,0.581,3.686,0.578,3.659,0.580,3.649,0.572,0.058,0.114
1,Dialog2Flow,dinamico,IVF,100,3.665,0.588,3.690,0.574,3.669,0.591,3.646,0.590,0.059,0.115
2,Dialog2Flow,dinamico,IVFPQ,100,3.647,0.629,3.659,0.629,3.658,0.632,3.629,0.649,0.063,0.123
3,Dialog2Flow,ema_alpha_0_1,HNSW,100,3.203,0.730,3.221,0.741,3.191,0.720,3.175,0.741,0.073,0.143
4,Dialog2Flow,ema_alpha_0_1,IVF,100,3.180,0.761,3.211,0.764,3.176,0.761,3.173,0.757,0.076,0.149
5,Dialog2Flow,ema_alpha_0_1,IVFPQ,100,3.181,0.755,3.225,0.767,3.182,0.750,3.162,0.748,0.075,0.148
6,Dialog2Flow,ema_alpha_0_2,HNSW,100,3.237,0.737,3.257,0.749,3.231,0.739,3.229,0.732,0.074,0.144
7,Dialog2Flow,ema_alpha_0_2,IVF,100,3.260,0.734,3.285,0.749,3.256,0.737,3.250,0.729,0.073,0.144
8,Dialog2Flow,ema_alpha_0_2,IVFPQ,100,3.326,0.674,3.353,0.685,3.323,0.675,3.319,0.677,0.067,0.132
9,Dialog2Flow,ema_alpha_0_3,HNSW,100,3.387,0.705,3.411,0.719,3.380,0.702,3.372,0.707,0.071,0.138


## 16. Comparación pareada vs dinámico anterior

In [ ]:
paired_scores = query_metrics.pivot_table(
    index=['embedding_base', 'index_type', 'query_order'],
    columns='variant',
    values='mss_at10',
).reset_index()

for baseline in ['estatico', 'dinamico']:
    if baseline not in paired_scores.columns:
        raise ValueError(
            f'Falta baseline {baseline}. Puede ser que todavía no esté evaluado. '
            f'Columnas disponibles: {paired_scores.columns.tolist()}'
        )

def variant_to_alpha(variant):
    if str(variant).startswith('ema_alpha_'):
        return float(str(variant).replace('ema_alpha_', '').replace('_', '.'))
    return np.nan

try:
    from scipy.stats import wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

def wilcoxon_safe(deltas):
    deltas = pd.Series(deltas).dropna()
    if SCIPY_AVAILABLE and len(deltas) > 0 and not np.allclose(deltas, 0):
        try:
            stat, p_value = wilcoxon(deltas)
            return stat, p_value
        except Exception:
            return np.nan, np.nan
    return np.nan, np.nan

rows = []
EPS = 1e-9
ema_variants = [v for v in VARIANTES_A_EVALUAR if v.startswith('ema_alpha_')]

for (embedding_base, index_type), group in paired_scores.groupby(['embedding_base', 'index_type'], sort=False):
    for variant in ['dinamico'] + ema_variants:
        if variant not in group.columns:
            continue
        cols_needed = ['estatico', 'dinamico', variant]
        sub = group.dropna(subset=cols_needed)
        if sub.empty:
            continue
        delta_vs_static = sub[variant] - sub['estatico']
        delta_vs_dynamic = sub[variant] - sub['dinamico']
        stat_static, p_static = wilcoxon_safe(delta_vs_static)
        stat_dynamic, p_dynamic = wilcoxon_safe(delta_vs_dynamic)
        rows.append({
            'embedding_base': embedding_base,
            'index_type': index_type,
            'variant': variant,
            'alpha': variant_to_alpha(variant),
            'n_queries': int(sub['query_order'].nunique()),
            'mean_static_mss_at10': float(sub['estatico'].mean()),
            'mean_dynamic_mss_at10': float(sub['dinamico'].mean()),
            'mean_variant_mss_at10': float(sub[variant].mean()),
            'mean_delta_vs_static': float(delta_vs_static.mean()),
            'median_delta_vs_static': float(delta_vs_static.median()),
            'sd_delta_vs_static': float(delta_vs_static.std()),
            'pct_variant_better_than_static': float((delta_vs_static > EPS).mean()),
            'pct_static_better_than_variant': float((delta_vs_static < -EPS).mean()),
            'wilcoxon_stat_vs_static': stat_static,
            'wilcoxon_p_vs_static': p_static,
            'mean_delta_vs_dynamic': float(delta_vs_dynamic.mean()),
            'median_delta_vs_dynamic': float(delta_vs_dynamic.median()),
            'sd_delta_vs_dynamic': float(delta_vs_dynamic.std()),
            'pct_variant_better_than_dynamic': float((delta_vs_dynamic > EPS).mean()),
            'pct_dynamic_better_than_variant': float((delta_vs_dynamic < -EPS).mean()),
            'wilcoxon_stat_vs_dynamic': stat_dynamic,
            'wilcoxon_p_vs_dynamic': p_dynamic,
        })

comparison_summary = pd.DataFrame(rows)
for col in comparison_summary.select_dtypes(include=['float']).columns:
    comparison_summary[col] = comparison_summary[col].round(4)

display(comparison_summary)

pivot_delta_vs_dynamic = comparison_summary.pivot_table(index='variant', columns='index_type', values='mean_delta_vs_dynamic')
pivot_pct_better_than_dynamic = comparison_summary.pivot_table(index='variant', columns='index_type', values='pct_variant_better_than_dynamic')

variant_order = ['dinamico'] + ema_variants
index_order = ['IVF', 'HNSW', 'IVFPQ']
pivot_delta_vs_dynamic = pivot_delta_vs_dynamic.reindex([v for v in variant_order if v in pivot_delta_vs_dynamic.index])
pivot_pct_better_than_dynamic = pivot_pct_better_than_dynamic.reindex([v for v in variant_order if v in pivot_pct_better_than_dynamic.index])
pivot_delta_vs_dynamic = pivot_delta_vs_dynamic[[c for c in index_order if c in pivot_delta_vs_dynamic.columns]]
pivot_pct_better_than_dynamic = pivot_pct_better_than_dynamic[[c for c in index_order if c in pivot_pct_better_than_dynamic.columns]]

print('Delta medio vs dinámico anterior')
display(pivot_delta_vs_dynamic)
print('% queries donde variante supera dinámico anterior')
display(pivot_pct_better_than_dynamic)

,embedding_base,index_type,variant,alpha,n_queries,mean_static_mss_at10,mean_dynamic_mss_at10,mean_variant_mss_at10,mean_delta_vs_static,median_delta_vs_static,...,pct_static_better_than_variant,wilcoxon_stat_vs_static,wilcoxon_p_vs_static,mean_delta_vs_dynamic,median_delta_vs_dynamic,sd_delta_vs_dynamic,pct_variant_better_than_dynamic,pct_dynamic_better_than_variant,wilcoxon_stat_vs_dynamic,wilcoxon_p_vs_dynamic
0,Dialog2Flow,HNSW,dinamico,NaN,100,3.293,3.661,3.661,0.368,0.30,...,0.28,899.5,0.0000,0.000,0.00,0.0000,0.00,0.00,NaN,NaN
1,Dialog2Flow,HNSW,ema_alpha_0_1,0.1,100,3.293,3.661,3.203,-0.090,0.10,...,0.44,2110.0,0.5279,-0.458,-0.40,0.7312,0.23,0.65,654.5,0.0000
2,Dialog2Flow,HNSW,ema_alpha_0_2,0.2,100,3.293,3.661,3.237,-0.056,0.10,...,0.43,2085.0,0.5779,-0.424,-0.30,0.6751,0.24,0.68,759.0,0.0000
3,Dialog2Flow,HNSW,ema_alpha_0_3,0.3,100,3.293,3.661,3.387,0.094,0.10,...,0.41,1899.5,0.3509,-0.274,-0.10,0.5899,0.30,0.58,963.0,0.0000
4,Dialog2Flow,HNSW,ema_alpha_0_4,0.4,100,3.293,3.661,3.577,0.284,0.30,...,0.30,1278.0,0.0008,-0.084,0.00,0.4718,0.35,0.49,1360.5,0.0581
5,Dialog2Flow,HNSW,ema_alpha_0_5,0.5,100,3.293,3.661,3.661,0.368,0.30,...,0.31,1010.5,0.0000,0.000,0.00,0.1933,0.22,0.24,538.0,0.9780
6,Dialog2Flow,HNSW,ema_alpha_0_6,0.6,100,3.293,3.661,3.776,0.483,0.40,...,0.21,741.0,0.0000,0.115,0.00,0.4988,0.45,0.38,1322.0,0.0557
7,Dialog2Flow,HNSW,ema_alpha_0_7,0.7,100,3.293,3.661,3.758,0.465,0.40,...,0.16,464.0,0.0000,0.097,0.00,0.5387,0.46,0.39,1491.5,0.1406
8,Dialog2Flow,HNSW,ema_alpha_0_8,0.8,100,3.293,3.661,3.639,0.346,0.20,...,0.18,622.0,0.0000,-0.022,0.00,0.5991,0.40,0.46,1685.5,0.4254
9,Dialog2Flow,HNSW,ema_alpha_0_9,0.9,100,3.293,3.661,3.414,0.121,0.10,...,0.28,1088.0,0.0045,-0.247,-0.20,0.6430,0.32,0.60,1255.0,0.0006


Delta medio vs dinámico anterior


index_type,IVF,HNSW,IVFPQ
variant,,,
dinamico,0.000,0.000,0.000
ema_alpha_0_1,-0.485,-0.458,-0.466
ema_alpha_0_2,-0.405,-0.424,-0.321
ema_alpha_0_3,-0.273,-0.274,-0.193
ema_alpha_0_4,-0.083,-0.084,-0.082
ema_alpha_0_5,0.006,0.000,-0.007
ema_alpha_0_6,0.132,0.115,0.067
ema_alpha_0_7,0.090,0.097,0.055
ema_alpha_0_8,-0.013,-0.022,-0.072


% queries donde variante supera dinámico anterior


index_type,IVF,HNSW,IVFPQ
variant,,,
dinamico,0.00,0.00,0.00
ema_alpha_0_1,0.29,0.23,0.28
ema_alpha_0_2,0.25,0.24,0.33
ema_alpha_0_3,0.29,0.30,0.36
ema_alpha_0_4,0.37,0.35,0.35
ema_alpha_0_5,0.26,0.22,0.38
ema_alpha_0_6,0.52,0.45,0.45
ema_alpha_0_7,0.49,0.46,0.48
ema_alpha_0_8,0.38,0.40,0.38


## 17. Tablas compactas

In [ ]:
pivot_mss_mean = summary_metrics.pivot_table(
    index='embedding_base',
    columns=['variant', 'index_type'],
    values='mean_semantic_score_global_at10',
)

pivot_mss_sd = summary_metrics.pivot_table(
    index='embedding_base',
    columns=['variant', 'index_type'],
    values='sd_semantic_score_global_at10',
)

ordered_cols = []
for variant in VARIANTES_A_EVALUAR:
    for idx in ['IVF', 'HNSW', 'IVFPQ']:
        if (variant, idx) in pivot_mss_mean.columns:
            ordered_cols.append((variant, idx))

pivot_mss_mean = pivot_mss_mean[ordered_cols]
pivot_mss_sd = pivot_mss_sd[[c for c in ordered_cols if c in pivot_mss_sd.columns]]
pivot_mss_mean_sd = pivot_mss_mean.copy().astype(object)

for col in pivot_mss_mean.columns:
    for row_idx in pivot_mss_mean.index:
        mean_val = pivot_mss_mean.loc[row_idx, col]
        sd_val = pivot_mss_sd.loc[row_idx, col]
        if pd.isna(mean_val):
            pivot_mss_mean_sd.loc[row_idx, col] = ''
        else:
            pivot_mss_mean_sd.loc[row_idx, col] = f'{mean_val:.3f} ± {sd_val:.3f}'

summary_alpha = summary_metrics.copy()
summary_alpha['alpha'] = summary_alpha['variant'].map(variant_to_alpha)
summary_alpha = summary_alpha[summary_alpha['variant'].astype(str).str.startswith('ema_alpha_')]

pivot_alpha_mss = summary_alpha.pivot_table(
    index='alpha',
    columns='index_type',
    values='mean_semantic_score_global_at10',
).sort_index()

print('MSS@10 — media')
display(pivot_mss_mean)
print('MSS@10 — media ± SD')
display(pivot_mss_mean_sd)
print('EMA: alpha vs MSS@10')
display(pivot_alpha_mss)

MSS@10 — media


variant        estatico               dinamico               ema_alpha_0_1  \
index_type          IVF   HNSW  IVFPQ      IVF   HNSW  IVFPQ           IVF   
embedding_base                                                               
Dialog2Flow       3.294  3.293  3.296    3.665  3.661  3.647          3.18   

variant                      ema_alpha_0_2  ... ema_alpha_0_6 ema_alpha_0_7  \
index_type       HNSW  IVFPQ           IVF  ...         IVFPQ           IVF   
embedding_base                              ...                               
Dialog2Flow     3.203  3.181          3.26  ...         3.714         3.755   

variant                      ema_alpha_0_8               ema_alpha_0_9         \
index_type       HNSW  IVFPQ           IVF   HNSW  IVFPQ           IVF   HNSW   
embedding_base                                                                  
Dialog2Flow     3.758  3.702         3.652  3.639  3.575         3.431  3.414   

variant                
index_type      IVFPQ  
embedding_base         
Dialog2Flow     3.491  

[1 rows x 33 columns]

MSS@10 — media ± SD


variant              estatico                                     dinamico  \
index_type                IVF           HNSW          IVFPQ            IVF   
embedding_base                                                               
Dialog2Flow     3.294 ± 0.803  3.293 ± 0.767  3.296 ± 0.742  3.665 ± 0.588   

variant                                       ema_alpha_0_1                 \
index_type               HNSW          IVFPQ            IVF           HNSW   
embedding_base                                                               
Dialog2Flow     3.661 ± 0.581  3.647 ± 0.629  3.180 ± 0.761  3.203 ± 0.730   

variant                        ema_alpha_0_2  ...  ema_alpha_0_6  \
index_type              IVFPQ            IVF  ...          IVFPQ   
embedding_base                                ...                  
Dialog2Flow     3.181 ± 0.755  3.260 ± 0.734  ...  3.714 ± 0.621   

variant         ema_alpha_0_7                                ema_alpha_0_8  \
index_type                IVF           HNSW          IVFPQ            IVF   
embedding_base                                                               
Dialog2Flow     3.755 ± 0.678  3.758 ± 0.668  3.702 ± 0.654  3.652 ± 0.697   

variant                                       ema_alpha_0_9                 \
index_type               HNSW          IVFPQ            IVF           HNSW   
embedding_base                                                               
Dialog2Flow     3.639 ± 0.709  3.575 ± 0.720  3.431 ± 0.776  3.414 ± 0.762   

variant                        
index_type              IVFPQ  
embedding_base                 
Dialog2Flow     3.491 ± 0.730  

[1 rows x 33 columns]

EMA: alpha vs MSS@10


index_type,HNSW,IVF,IVFPQ
alpha,,,
0.1,3.203,3.180,3.181
0.2,3.237,3.260,3.326
0.3,3.387,3.392,3.454
0.4,3.577,3.582,3.565
0.5,3.661,3.671,3.640
0.6,3.776,3.797,3.714
0.7,3.758,3.755,3.702
0.8,3.639,3.652,3.575
0.9,3.414,3.431,3.491


## 18. Ejemplo con scores

In [ ]:
def mostrar_vecinos_con_scores(df_scores, variante='ema_alpha_0_5', index_type='HNSW', query_order=0, top_n=10):
    sub = df_scores[
        (df_scores['embedding_base'] == EMBEDDING_LABEL) &
        (df_scores['variant'] == variante) &
        (df_scores['index_type'] == index_type) &
        (df_scores['query_order'] == query_order)
    ].sort_values('neighbor_rank').head(top_n)
    if sub.empty:
        print('No hay registros para ese filtro.')
        return
    first = sub.iloc[0]
    display(Markdown(f'### Consulta `{query_order}` — {variante} / {index_type}'))
    display(Markdown('**Situación de consulta:**'))
    print(first['query_context'])
    cols = [
        'neighbor_rank',
        'distance',
        'overall_similarity',
        'semantic_similarity',
        'functional_similarity',
        'memory_usefulness',
        'neighbor_utterance',
        'brief_reason',
    ]
    display(sub[cols])

mostrar_vecinos_con_scores(df_scores, variante='ema_alpha_0_5', index_type='HNSW', query_order=0)

### Consulta `0` — ema_alpha_0_5 / HNSW

**Situación de consulta:**

[5] system: And, what's the city you want for your departure?
[6] user: I want to fly out of Atlanta, GA.
[7] system: Okay. I'm showing 3 flight options for you, including a flight on Delta Airlines with 0 stops, departing at 6:50 am for $260.


,neighbor_rank,distance,overall_similarity,semantic_similarity,functional_similarity,memory_usefulness,neighbor_utterance,brief_reason
15000,1,6.906024e-14,5,5,5,5,"Okay. I'm showing 3 flight options for you, in...",Exact match in dialogue content and function.
15001,2,1.018683e-01,4,4,4,4,"I want to fly out of Atlanta, GA.","Same departure city question and answer, with ..."
15002,3,1.018683e-01,4,4,4,4,"I want to fly out of Atlanta, GA.","Same as neighbor 2, similar dialogue flow and ..."
15003,4,1.146481e-01,4,4,4,4,What's the destination airport and arrival tim...,"Same departure city info and system response, ..."
15004,5,1.146481e-01,4,4,4,4,What's the destination airport and arrival tim...,"Same as neighbor 4, similar dialogue flow and ..."
15005,6,1.201017e-01,3,3,3,3,"I have locate 1 flight with Alaska Airlines, i...",Similar task of finding flights from a departu...
15006,7,1.274708e-01,2,2,2,2,I've found 5 flights that'll fit your needs. T...,Different focus on flight date and different c...
15007,8,1.394682e-01,3,3,3,3,I have 4 flights that match. There's a 0 stop ...,Similar departure city question and flight opt...
15008,9,1.423699e-01,2,2,2,2,I found 4 flights for you. Would you like to f...,Different departure city and additional destin...
15009,10,1.478643e-01,2,2,2,2,"Ok, I found an American Airlines flight with 1...",Different departure city and airline preferenc...


## 19. Exportación final

In [ ]:
fecha_hora_arg = datetime.now(ZoneInfo('America/Argentina/Buenos_Aires')).strftime('%Y%m%d_%H%M%S_ARG')

retrieval_csv = FINAL_DIR / f'retrieval_top10_textual_ema_dialog2flow_{fecha_hora_arg}.csv'
scores_csv = FINAL_DIR / f'llm_semantic_scores_pairs_ema_dialog2flow_{fecha_hora_arg}.csv'
query_metrics_csv = FINAL_DIR / f'llm_semantic_query_metrics_ema_dialog2flow_{fecha_hora_arg}.csv'
summary_csv = FINAL_DIR / f'llm_semantic_summary_ema_dialog2flow_{fecha_hora_arg}.csv'
comparison_csv = FINAL_DIR / f'ema_vs_dynamic_summary_{fecha_hora_arg}.csv'
progress_csv = FINAL_DIR / f'progress_ema_dialog2flow_{fecha_hora_arg}.csv'
pivot_delta_csv = FINAL_DIR / f'table_delta_ema_vs_dynamic_pivot_{fecha_hora_arg}.csv'
pivot_pct_csv = FINAL_DIR / f'table_pct_ema_better_than_dynamic_pivot_{fecha_hora_arg}.csv'
pivot_mean_csv = FINAL_DIR / f'table_mss_at10_mean_pivot_ema_{fecha_hora_arg}.csv'
pivot_sd_csv = FINAL_DIR / f'table_mss_at10_sd_pivot_ema_{fecha_hora_arg}.csv'
pivot_mean_sd_csv = FINAL_DIR / f'table_mss_at10_mean_sd_pivot_ema_{fecha_hora_arg}.csv'
pivot_alpha_csv = FINAL_DIR / f'table_alpha_mss_at10_ema_{fecha_hora_arg}.csv'
pivot_mean_sd_latex = FINAL_DIR / f'table_mss_at10_mean_sd_pivot_ema_{fecha_hora_arg}.tex'

df_retrieval.to_csv(retrieval_csv, index=False)
df_scores.to_csv(scores_csv, index=False)
query_metrics.to_csv(query_metrics_csv, index=False)
summary_metrics.to_csv(summary_csv, index=False)
comparison_summary.to_csv(comparison_csv, index=False)
progress_df.to_csv(progress_csv, index=False)
pivot_delta_vs_dynamic.to_csv(pivot_delta_csv)
pivot_pct_better_than_dynamic.to_csv(pivot_pct_csv)
pivot_mss_mean.to_csv(pivot_mean_csv)
pivot_mss_sd.to_csv(pivot_sd_csv)
pivot_mss_mean_sd.to_csv(pivot_mean_sd_csv)
pivot_alpha_mss.to_csv(pivot_alpha_csv)

with open(pivot_mean_sd_latex, 'w', encoding='utf-8') as f:
    f.write(pivot_mss_mean_sd.to_latex())

latest_map = {
    retrieval_csv: 'retrieval_top10_textual_ema_dialog2flow_latest.csv',
    scores_csv: 'llm_semantic_scores_pairs_ema_dialog2flow_latest.csv',
    query_metrics_csv: 'llm_semantic_query_metrics_ema_dialog2flow_latest.csv',
    summary_csv: 'llm_semantic_summary_ema_dialog2flow_latest.csv',
    comparison_csv: 'ema_vs_dynamic_summary_latest.csv',
    progress_csv: 'progress_ema_dialog2flow_latest.csv',
    pivot_delta_csv: 'table_delta_ema_vs_dynamic_pivot_latest.csv',
    pivot_pct_csv: 'table_pct_ema_better_than_dynamic_pivot_latest.csv',
    pivot_mean_csv: 'table_mss_at10_mean_pivot_ema_latest.csv',
    pivot_sd_csv: 'table_mss_at10_sd_pivot_ema_latest.csv',
    pivot_mean_sd_csv: 'table_mss_at10_mean_sd_pivot_ema_latest.csv',
    pivot_alpha_csv: 'table_alpha_mss_at10_ema_latest.csv',
}

for src, latest_name in latest_map.items():
    shutil.copy(src, FINAL_DIR / latest_name)

print('Archivos finales guardados en:')
print(FINAL_DIR)

Archivos finales guardados en:
/content/drive/MyDrive/Doctorado/Cursos/ANN/TF/resultados/semantic_llm/version_4/final_tables
